# Import dos dados e bibliotecas

In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

import pyarrow
np.random.seed(21)

In [12]:
df_caminhao = pd.read_parquet('datasets/datasets_tratados/df_caminhao_modelagem.parquet', engine="pyarrow")

# Tratamento dos dados para modelagem

In [13]:
df_caminhao.columns

Index(['Data_Evento', 'Dia', 'TAG', 'Tag_Frota', 'Tipo', 'Nome_Operador_Anon',
       'Alarme', 'Criticidade', 'Valor', 'Classe',
       ...
       'EWMA_72h_Dim_22', 'EWMA_72h_Dim_23', 'EWMA_72h_Dim_24',
       'EWMA_72h_Dim_25', 'EWMA_72h_Dim_26', 'EWMA_72h_Dim_27',
       'EWMA_72h_Dim_28', 'EWMA_72h_Dim_29', 'EWMA_72h_Dim_30',
       'EWMA_72h_Dim_31'],
      dtype='str', length=119)

In [14]:
print("Calculando os Deltas de Degradação (Derivadas Temporais)...")

cat_cols = [
    'Situacao_Operacional',
    'Nome_Operador_Anon',
    'Criticidade',
    'Turno',
]

for col in cat_cols:
    df_caminhao[col] = df_caminhao[col].astype('category')

# Dicionário temporário para armazenar as novas colunas 
# (Isso evita o erro de 'PerformanceWarning: DataFrame is highly fragmented' do Pandas)
novas_colunas = {}
colunas_vetor = [f'Dim_{i}' for i in range(32)]

# O loop passa pelas 32 dimensões (Dim_0 até Dim_31)
for col in colunas_vetor:
    
    # 1. Delta Curto (Agudo vs Turno): O que mudou nas últimas 2h em relação às últimas 12h?
    novas_colunas[f'Delta_2h_12h_{col}'] = df_caminhao[f'EWMA_2h_{col}'] - df_caminhao[f'EWMA_12h_{col}']
    
    # 2. Delta Longo (Agudo vs Histórico): O que mudou nas últimas 2h em relação aos últimos 3 dias?
    novas_colunas[f'Delta_2h_72h_{col}'] = df_caminhao[f'EWMA_2h_{col}'] - df_caminhao[f'EWMA_72h_{col}']

# Transforma o dicionário em DataFrame e cola no principal de uma única vez
df_deltas = pd.DataFrame(novas_colunas)
df_caminhao = pd.concat([df_caminhao, df_deltas], axis=1)

# Atualizamos a lista final de features para o LightGBM (pegando as EWMAs e os Deltas)
colunas_ewma_deltas = [col for col in df_caminhao.columns if col.startswith('EWMA_') or col.startswith('Delta_')]
features = colunas_ewma_deltas + cat_cols + ['Dia', 'Mes', 'Tempo_Turno', 'Tempo_Situacao']
target = 'Minutos_Ate_Proximo_Dont_Go'

Calculando os Deltas de Degradação (Derivadas Temporais)...


# Separação dos dados

In [15]:
# =====================================================================
# 1. SORTEIO DO HOLDOUT (MÁQUINAS INÉDITAS)
# =====================================================================
# Define a semente para garantir que o sorteio seja reprodutível

tags_unicas = df_caminhao['TAG'].unique()
qtd_holdout = int(len(tags_unicas) * 0.15) # 15% dos caminhões para o teste cego

tags_holdout = np.random.choice(tags_unicas, size=qtd_holdout, replace=False)

# Separa a base de Holdout Absoluto e a Base de Desenvolvimento (Treino + Validação)
df_test = df_caminhao[df_caminhao['TAG'].isin(tags_holdout)].copy()
df_caminhao = df_caminhao[~df_caminhao['TAG'].isin(tags_holdout)].copy()

print(f"Caminhões no Holdout (Nunca vistos): {len(tags_holdout)}")
print(f"Caminhões no Desenvolvimento: {len(tags_unicas) - len(tags_holdout)}")

Caminhões no Holdout (Nunca vistos): 4
Caminhões no Desenvolvimento: 24


In [16]:
# =====================================================================
# 2. SPLIT TEMPORAL POR VEÍCULO (TREINO E VALIDAÇÃO)
# =====================================================================
# Garante que está ordenado no tempo
df_caminhao = df_caminhao.sort_values(by=['TAG', 'Data_Evento'])

# Calcula o percentil temporal de cada log dentro da história do seu respectivo caminhão
df_caminhao['Ordem_Tempo'] = df_caminhao.groupby('TAG')['Data_Evento'].rank(method='first')
df_caminhao['Total_Logs_TAG'] = df_caminhao.groupby('TAG')['TAG'].transform('count')
df_caminhao['Percentil_Tempo'] = df_caminhao['Ordem_Tempo'] / df_caminhao['Total_Logs_TAG']

# Corta em 80% (Passado) e 20% (Futuro) para cada caminhão
df_train = df_caminhao[df_caminhao['Percentil_Tempo'] <= 0.80].copy()
df_val = df_caminhao[df_caminhao['Percentil_Tempo'] > 0.80].copy()

# Limpa as colunas auxiliares que criamos
df_train.drop(columns=['Ordem_Tempo', 'Total_Logs_TAG', 'Percentil_Tempo'], inplace=True)
df_val.drop(columns=['Ordem_Tempo', 'Total_Logs_TAG', 'Percentil_Tempo'], inplace=True)

print(f"\nLinhas de Treino (Passado): {len(df_train):,}")
print(f"Linhas de Validação (Futuro): {len(df_val):,}")
print(f"Linhas de Holdout (Teste Cego): {len(df_test):,}")


Linhas de Treino (Passado): 502,596
Linhas de Validação (Futuro): 125,662
Linhas de Holdout (Teste Cego): 212,400


In [17]:
# Treino e Validação (Usado para guiar o aprendizado do modelo)
X_train, y_train = df_train[features], df_train[target]
X_val, y_val = df_val[features], df_val[target]

# Holdout (Usado apenas no final para julgar a performance real)
X_test, y_test = df_test[features], df_test[target]

In [18]:
df_caminhao['Minutos_Ate_Proximo_Dont_Go'].describe()

count    628258.000000
mean       4013.276658
std        7406.728032
min           0.016667
25%         135.440979
50%        1156.209750
75%        4385.717217
max       71812.236900
Name: Minutos_Ate_Proximo_Dont_Go, dtype: float64

# Modelagem

In [19]:
TETO_MINUTOS = 3 * 24 * 60 

# Qualquer tempo maior que 3 dias vira exatos 3 dias. 
# O modelo aprende: "dados calmos = prever 4320"
y_train_piecewise = np.clip(y_train, a_min=0, a_max=TETO_MINUTOS)
y_val_piecewise = np.clip(y_val, a_min=0, a_max=TETO_MINUTOS)

In [20]:
modelo_cb = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.0003,
    
    # --- ALAVANCAS ANTI-OVERFITTING ---
    depth=4,                    # Árvores mais rasas e generalistas
    # l2_leaf_reg=100,             # Punição severa contra memorização (Padrão é 3)
    rsm=0.5,                    # Usa apenas 70% das features por árvore
    subsample=0.7,              # Usa apenas 80% dos dados por árvore
    bootstrap_type='Bernoulli', # Obrigatório para usar o subsample
    # random_strength=1.5,        # Adiciona ruído nas divisões para evitar overfitting
    # ----------------------------------
    
    loss_function='MAE',
    random_seed=42,
    od_type='Iter',
    od_wait=150, # Damos mais paciência para ele convergir com a regularização
    verbose=100
)

modelo_cb.fit(
    X_train, y_train_piecewise,
    cat_features=cat_cols,
    eval_set=(X_val, y_val_piecewise),
    use_best_model=True
)

0:	learn: 1493.3509579	test: 1770.7504698	best: 1770.7504698 (0)	total: 83.6ms	remaining: 13m 55s
100:	learn: 1475.7203688	test: 1756.1743946	best: 1756.1743946 (100)	total: 6.5s	remaining: 10m 37s
200:	learn: 1458.9923498	test: 1742.1581837	best: 1742.1581837 (200)	total: 12.8s	remaining: 10m 22s
300:	learn: 1443.0354090	test: 1729.3960721	best: 1729.3960721 (300)	total: 19.1s	remaining: 10m 14s
400:	learn: 1427.9153530	test: 1717.4321225	best: 1717.4321225 (400)	total: 25.5s	remaining: 10m 9s
500:	learn: 1413.3483289	test: 1706.0244130	best: 1706.0244130 (500)	total: 31.8s	remaining: 10m 3s
600:	learn: 1399.5103479	test: 1695.3221956	best: 1695.3221956 (600)	total: 38.2s	remaining: 9m 57s
700:	learn: 1386.2825741	test: 1685.2314480	best: 1685.2314480 (700)	total: 44.6s	remaining: 9m 51s
800:	learn: 1373.6785715	test: 1675.6427628	best: 1675.6427628 (800)	total: 51s	remaining: 9m 45s
900:	learn: 1361.5420617	test: 1666.7214507	best: 1666.7214507 (900)	total: 57.5s	remaining: 9m 40s
10

CatBoostRegressor(bootstrap_type='Bernoulli', depth=4, iterations=10000, learning_rate=0.0003, loss_function='MAE', od_type='Iter', od_wait=150, random_seed=42, rsm=0.5, subsample=0.7, verbose=100)

In [21]:
# ==========================================
# 3. AVALIAÇÃO FOCADA (A PROVA DE FOGO)
# ==========================================
# Fazemos a previsão no conjunto de validação (ou teste)
y_pred = modelo_cb.predict(X_val)

# Criamos as máscaras para separar a Zona de Paz da Zona de Perigo
mascara_perigo = y_val <= (48 * 60) # Caminhões a menos de 48h da falha (Realidade)
mascara_paz = y_val > (48 * 60)     # Caminhões a mais de 48h da falha (Realidade)

# Calculamos o erro APENAS quando o caminhão está realmente quebrando
mae_perigo = mean_absolute_error(y_val[mascara_perigo], y_pred[mascara_perigo])

print("\n--- PERFORMANCE OPERACIONAL ---")
print(f"🎯 MAE na ZONA DE PERIGO (< 48h): {mae_perigo:.2f} minutos ({mae_perigo/60:.2f} horas)")

# Como o modelo se comporta na paz? (Deveria prever valores altos, próximos ao teto)
media_paz_prevista = np.mean(y_pred[mascara_paz])
print(f"🛡️ Previsão Média na ZONA DE PAZ: {media_paz_prevista:.2f} minutos ({media_paz_prevista/60:.2f} horas)")


--- PERFORMANCE OPERACIONAL ---
🎯 MAE na ZONA DE PERIGO (< 48h): 1117.77 minutos (18.63 horas)
🛡️ Previsão Média na ZONA DE PAZ: 2098.38 minutos (34.97 horas)
